In [1]:
import pandas as pd
import numpy as np
import re
import wordninja
import os
import string
import langdetect
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from deep_translator import GoogleTranslator  # Conditional import
#from swifter import swifter #pip install swifter

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
#load dataset
file_path = 'C:\\Users\\Lish Ai Labs\\Desktop\\Simba\\wired_articles_business_social-media.csv'
df = pd.read_csv(file_path)
print(df.head())


                                               title  \
0   How Meta Tried to Lure TikTok Users to Instagram   
1  RedNote Recruited US Influencers to Promote Ap...   
2                      TikTok Is Already Back Online   
3  It’s Not Just TikTok: These Other ByteDance Ap...   
4  TikTok Is Unavailable in the US—and Gone From ...   

                                                 url  \
0  https://www.wired.com/story/how-meta-tried-to-...   
1  https://www.wired.com/story/rednote-is-asking-...   
2        https://www.wired.com/story/tiktok-is-back/   
3  https://www.wired.com/story/bytedance-tiktok-b...   
4  https://www.wired.com/story/tiktok-ban-officia...   

                       author            rubric  \
0             Louise Matsakis  Seize the Moment   
1                Makena Kelly      Fame Farming   
2                Zoë Schiffer          Miss Me?   
3  Zeyi Yang and Andrew Couts    bytes the dust   
4             Louise Matsakis    App Apocalypse   

                   

In [3]:
print(df.dtypes)


title           object
url             object
author          object
rubric          object
full_content    object
dtype: object


In [4]:
# checking for missing values
missing_values = df.isnull()
for column in missing_values.columns.values.tolist():
    print (missing_values[column].value_counts())
    print("")



title
False    1019
Name: count, dtype: int64

url
False    1019
Name: count, dtype: int64

author
False    1019
Name: count, dtype: int64

rubric
False    1019
Name: count, dtype: int64

full_content
False    1016
True        3
Name: count, dtype: int64



In [5]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

number of duplicate rows:  (3, 5)


In [6]:
#dropping duplicates
df = df.drop_duplicates()
print("number of duplicate rows: ", df.duplicated())

number of duplicate rows:  0       False
1       False
2       False
3       False
4       False
        ...  
1014    False
1015    False
1016    False
1017    False
1018    False
Length: 1016, dtype: bool


In [7]:
# Drop rows where BOTH columns 'full_content' and 'title' are empty
df1 = df.copy() 
df_cleaned = df1.dropna(subset=['full_content', 'title'], how='all')

print(df_cleaned.shape)  
print(df_cleaned.isnull().sum()) 

(1016, 5)
title           0
url             0
author          0
rubric          0
full_content    3
dtype: int64


In [8]:
# having only 2 rows with missing we can replace them with Not Available
df1 = df.copy()
df_cleaned = df1.copy()  

# Show the count of null values before processing
print(df_cleaned.isnull().sum())

# Function to extract first 25 words from 'summary' if 'full_content' is NaN
def get_full_content(row):
    if pd.isna(row['full_content']):  
        if 'summary' in row and pd.notna(row['summary']): 
            text = row['summary']
            if isinstance(text, str):  
                words = text.split()
                return ' '.join(words[:25]) if len(words) > 25 else text
            else:
                return 'Not Available'  
        else:
            return 'Not Available' 
    else:
        return row['full_content']  


df_cleaned['full_content'] = df_cleaned.apply(get_full_content, axis=1)
df_cleaned['full_content'] = df_cleaned['full_content'].fillna("Not Available")



print(df_cleaned.isnull().sum())
print(df_cleaned.head())

title           0
url             0
author          0
rubric          0
full_content    3
dtype: int64
title           0
url             0
author          0
rubric          0
full_content    0
dtype: int64
                                               title  \
0   How Meta Tried to Lure TikTok Users to Instagram   
1  RedNote Recruited US Influencers to Promote Ap...   
2                      TikTok Is Already Back Online   
3  It’s Not Just TikTok: These Other ByteDance Ap...   
4  TikTok Is Unavailable in the US—and Gone From ...   

                                                 url  \
0  https://www.wired.com/story/how-meta-tried-to-...   
1  https://www.wired.com/story/rednote-is-asking-...   
2        https://www.wired.com/story/tiktok-is-back/   
3  https://www.wired.com/story/bytedance-tiktok-b...   
4  https://www.wired.com/story/tiktok-ban-officia...   

                       author            rubric  \
0             Louise Matsakis  Seize the Moment   
1                M

In [9]:
# checking if there are any missing values left
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    1016
Name: count, dtype: int64

url
False    1016
Name: count, dtype: int64

author
False    1016
Name: count, dtype: int64

rubric
False    1016
Name: count, dtype: int64

full_content
False    1016
Name: count, dtype: int64



In [10]:
# Remove \n from 'full_content'
df_cleaned["full_content"] = df_cleaned["full_content"].str.replace("\n", " ", regex=True)

# Columns to check for newline characters
columns_to_check = ['full_content', 'title', 'url', 'author', 'rubric']

# Checking for \n after replacing. 
for col in columns_to_check:
    if col in df_cleaned.columns: 
        num_with_newline = df_cleaned[col].str.contains('\n').sum()
        print(f"Number of rows in '{col}' with newline characters: {num_with_newline}")
    else:
        print(f"Column '{col}' not found in DataFrame.")

print(df_cleaned.head())

Number of rows in 'full_content' with newline characters: 0
Number of rows in 'title' with newline characters: 0
Number of rows in 'url' with newline characters: 0
Number of rows in 'author' with newline characters: 0
Number of rows in 'rubric' with newline characters: 0
                                               title  \
0   How Meta Tried to Lure TikTok Users to Instagram   
1  RedNote Recruited US Influencers to Promote Ap...   
2                      TikTok Is Already Back Online   
3  It’s Not Just TikTok: These Other ByteDance Ap...   
4  TikTok Is Unavailable in the US—and Gone From ...   

                                                 url  \
0  https://www.wired.com/story/how-meta-tried-to-...   
1  https://www.wired.com/story/rednote-is-asking-...   
2        https://www.wired.com/story/tiktok-is-back/   
3  https://www.wired.com/story/bytedance-tiktok-b...   
4  https://www.wired.com/story/tiktok-ban-officia...   

                       author            rubric  \
0  

In [11]:
# normalizing text
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_dataframe(df, exclude_columns=['url']):

    for col in df.select_dtypes(include='object').columns:  
        if col not in exclude_columns: 
            try:
                df[col] = df[col].astype(str).apply(normalize_text)
                print(f"Column '{col}' normalized.")
            except Exception as e:
                print(f"Error normalizing column '{col}': {e}")
        else:
            print(f"Skipping normalization for column '{col}'.")
    return df

try:
    df = pd.read_csv(file_path)  

    # Normalize the entire DataFrame
    df = normalize_dataframe(df)

    # Print to check the results
    print(df_cleaned.head())

except FileNotFoundError:
    print("Error: CSV file not found.  Make sure the file path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

Column 'title' normalized.
Skipping normalization for column 'url'.
Column 'author' normalized.
Column 'rubric' normalized.
Column 'full_content' normalized.
                                               title  \
0   How Meta Tried to Lure TikTok Users to Instagram   
1  RedNote Recruited US Influencers to Promote Ap...   
2                      TikTok Is Already Back Online   
3  It’s Not Just TikTok: These Other ByteDance Ap...   
4  TikTok Is Unavailable in the US—and Gone From ...   

                                                 url  \
0  https://www.wired.com/story/how-meta-tried-to-...   
1  https://www.wired.com/story/rednote-is-asking-...   
2        https://www.wired.com/story/tiktok-is-back/   
3  https://www.wired.com/story/bytedance-tiktok-b...   
4  https://www.wired.com/story/tiktok-ban-officia...   

                       author            rubric  \
0             Louise Matsakis  Seize the Moment   
1                Makena Kelly      Fame Farming   
2              

In [12]:
# splitting words
df_cleaned = pd.read_csv(file_path)
print("DataFrame loaded successfully.") 

# Check for required columns *after* successful loading
columns_to_split = ['full_content', 'title', 'rubric']

# Apply word splitting to the specified columns
for col in columns_to_split:
    if col in df_cleaned.columns: #Only apply to columns that exist in the dataframe
        try:
            df_cleaned[col] = df_cleaned[col].astype(str).apply(lambda x: " ".join(wordninja.split(x)))
            print(f"Word splitting applied to column '{col}'.")
        except Exception as e:
            print(f"An error occurred during word splitting on column '{col}': {e}")
    else:
        print(f"Column '{col}' not found in the DataFrame")

print("DataFrame processing completed (with potential errors handled).")

DataFrame loaded successfully.
Word splitting applied to column 'full_content'.
Word splitting applied to column 'title'.
Word splitting applied to column 'rubric'.
DataFrame processing completed (with potential errors handled).


In [13]:
df_cleaned['url'] = np.where(df_cleaned['url'].isnull() | (df_cleaned['url'] == ''), df['url'], df_cleaned['url'])

df_cleaned.head()


,title,url,author,rubric,full_content
0,How Meta Tried to Lure Tik Tok Users to Insta ...,https://www.wired.com/story/how-meta-tried-to-...,Louise Matsakis,Seize the Moment,It was an opportunity too good for Meta to ign...
1,Red Note Recruited US Influence rs to Promote ...,https://www.wired.com/story/rednote-is-asking-...,Makena Kelly,Fame Farming,As Tik Tok s future hangs in the balance Xiao ...
2,Tik Tok Is Already Back Online,https://www.wired.com/story/tiktok-is-back/,Zoë Schiffer,Miss Me,Less than 24 hours after going dark Tik Tok sa...
3,It s Not Just Tik Tok These Other Byte Dance A...,https://www.wired.com/story/bytedance-tiktok-b...,Zeyi Yang and Andrew Couts,bytes the dust,Tik Tok is no longer available in the United S...
4,Tik Tok Is Unavailable in the US and Gone From...,https://www.wired.com/story/tiktok-ban-officia...,Louise Matsakis,App Apocalypse,For the first time in internet history the Uni...


In [14]:
# Define the function to remove URLs
def remove_url(text):
    """Removes URLs from a ."""
    if isinstance(text, str):
        return re.sub(r'https?://\S+|www\.\S+', '', text).strip() 
    return text 

def remove_punctuation(text):
    """Removes punctuation from a ."""
    if isinstance(text, str):
        return re.sub(r'[^\w\s]', '', text).strip() 
    return text 

def clean_dataframe(df, columns):
    """Cleans specified columns in a Pandas DataFrame by removing URLs and punctuation."""
    for col in columns:
        if col in df.columns: 
            if df[col].dtype == 'object':
                df[col] = df[col].astype(str).apply(remove_url)
                df[col] = df[col].astype(str).apply(remove_punctuation)
                print(f"Column '{col}' cleaned.")
            else:
                print(f"Skipping column '{col}' because it's not an object/ data type.")
        else:
            print(f"Column '{col}' not found in DataFrame.")

try:
    file_path = 'C:\\Users\\Lish Ai Labs\\Desktop\\Simba\\wired_articles - wired_articles.csv'
    df_cleaned = pd.read_csv(file_path) 

    # Specify the columns to be cleaned
    columns_to_clean = ['full_content', 'title', 'rubric']

    # Clean the DataFrame
    clean_dataframe(df_cleaned, columns_to_clean)

    # Display the first few rows to verify the changes
    print(df_cleaned.head().to_string()) 
    print("\nDataFrame Info:") 
    print(df_cleaned.info())

except FileNotFoundError:
    print("Error: CSV file not found. Please ensure the file path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

Column 'full_content' cleaned.
Column 'title' cleaned.
Column 'rubric' cleaned.
                                                                              title                                                                                    url            author                 rubric                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [15]:
# Define Cleaning Function (and Tokenization):
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = str(text).lower()
    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words and w.isalnum()]

    lemmatizer = WordNetLemmatizer() 
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return tokens


text_columns = df.select_dtypes(include=['object']).columns
print(f"Identified text columns: {text_columns}")

for column in text_columns:
    df[column + '_cleaned'] = df[column].apply(clean_text)

print(df_cleaned.head())


Identified text columns: Index(['title', 'url', 'author', 'rubric', 'full_content'], dtype='object')
                                               title  \
0  Elon Musks Man in the Treasury Is Still Holdin...   
1                 DOGEs Website Is Just One Big X Ad   
2                           DOGEs Race to the Bottom   
3  The GSA Plans to Sell Hundreds of Its Federal ...   
4  Former Palantir and Elon Musk Associates Are T...   

                                                 url            author  \
0  https://www.wired.com/story/musk-krause-treasu...  Vittoria Elliott   
1  https://www.wired.com/story/doge-website-is-ju...     David Gilbert   
2  https://www.wired.com/story/doge-elon-musk-fas...     Brian Barrett   
3  https://www.wired.com/story/gsa-sell-governmen...       Leah Feiger   
4  https://www.wired.com/story/elon-musk-palantir...      Makena Kelly   

                  rubric                                       full_content  
0           Moonlighting  This morning 

In [16]:
# saving the cleaned dataset
 
folder_path = os.path.join(os.path.expanduser("~"), "Desktop", "Simba") 

file_name = "cleaned_wired_articles_business_social-media.csv"
file_path = os.path.join(folder_path, file_name)

df.to_csv(file_path, index=False, encoding='utf-8')

print(f"Cleaned data saved to: {file_path}")

Cleaned data saved to: C:\Users\Lish Ai Labs\Desktop\Simba\cleaned_wired_articles_business_social-media.csv
